# Moduł 1: FastAPI - Wprowadzenie i Routing

**Ćwiczenie:** #1 - FastAPI Hello World + Routing

---

## Spis treści

1. [Wprowadzenie](#1-wprowadzenie)
2. [Kluczowe koncepcje](#2-kluczowe-koncepcje)
   - [Instalacja i pierwsze uruchomienie](#21-instalacja-i-pierwsze-uruchomienie)
   - [Path Operations (Routing)](#22-path-operations-routing)
   - [Path Parameters](#23-path-parameters-parametry-ścieżki)
   - [Query Parameters](#24-query-parameters-parametry-zapytania)
   - [Mieszanie Path i Query Parameters](#25-mieszanie-path-i-query-parameters)
   - [Kolejność deklaracji routes](#26-kolejność-deklaracji-routes-ważne)
   - [Zwracanie odpowiedzi](#27-zwracanie-odpowiedzi)
   - [Dokumentacja w Swagger](#28-dokumentacja-w-swagger-docstrings)
3. [Best Practices](#3-best-practices)
4. [Demo Live Coding](#4-demo-live-coding)
5. [Przygotowanie do ćwiczenia](#5-przygotowanie-do-ćwiczenia)
6. [Dodatkowe materiały](#6-dodatkowe-materiały)

---

## 1. Wprowadzenie

### Po co ten temat?

FastAPI to nowoczesny framework do budowy REST API w Pythonie, który:
- Jest **szybki** - porównywalny z Node.js i Go (dzięki async)
- Jest **łatwy w użyciu** - minimalna ilość kodu, intuicyjna składnia
- Jest **bezpieczny** - automatyczna walidacja danych dzięki Pydantic
- **Generuje dokumentację** automatycznie (Swagger UI)

### Gdzie to użyjemy w praktyce?

- Budowa API dla aplikacji webowych i mobilnych
- Mikroserwisy i systemy rozproszone
- Machine Learning inference endpoints
- Data pipelines (ETL) z HTTP interface
- Szybkie prototypy i MVP

**Dlaczego FastAPI zamiast Flask/Django?**
- **Flask:** Prostszy, ale bez walidacji i dokumentacji out-of-the-box
- **Django REST Framework:** Potężny, ale cięższy i bardziej opinionated
- **FastAPI:** Nowoczesny, szybki, minimalny boilerplate, świetna dokumentacja

## 2. Kluczowe koncepcje

### 2.1. Instalacja i pierwsze uruchomienie

**Instalacja:**

In [ ]:
# Instalacja FastAPI i Uvicorn
# Uruchom w terminalu:
# pip install fastapi uvicorn[standard]

- **FastAPI** - sam framework
- **Uvicorn** - ASGI server (serwer aplikacji, odpowiednik Gunicorn dla async)

**Pierwszy program (`main.py`):**

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def read_root():
    return {"message": "Hello World"}

**Uruchomienie:**

```bash
uvicorn main:app --reload
```

- `main` - nazwa pliku (bez .py)
- `app` - nazwa zmiennej FastAPI()
- `--reload` - auto-restart przy zmianach (tylko development!)

**Co się dzieje?**
1. Uvicorn startuje serwer na `http://localhost:8000`
2. FastAPI automatycznie generuje:
   - **Swagger UI**: `http://localhost:8000/docs` (interaktywna dokumentacja)
   - **ReDoc**: `http://localhost:8000/redoc` (alternatywna dokumentacja)
   - **OpenAPI schema**: `http://localhost:8000/openapi.json` (JSON schema)

### 2.2. Path Operations (Routing)

**Path operation** = połączenie HTTP method + path + funkcja

#### Podstawowe HTTP methods:

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/items")
def get_items():
    """Pobierz listę elementów"""
    return {"items": []}

@app.post("/items")
def create_item():
    """Stwórz nowy element"""
    return {"message": "Item created"}

@app.put("/items/{item_id}")
def update_item(item_id: int):
    """Zaktualizuj element"""
    return {"message": f"Item {item_id} updated"}

@app.delete("/items/{item_id}")
def delete_item(item_id: int):
    """Usuń element"""
    return {"message": f"Item {item_id} deleted"}

**Kluczowe zasady:**
- Funkcja może mieć dowolną nazwę (konwencja: czasownik + rzeczownik)
- Kolejność dekoratorów ma znaczenie (pierwsze dopasowanie wygrywa!)
- FastAPI automatycznie serializuje do JSON

### 2.3. Path Parameters (parametry ścieżki)

**Definicja:**
Dynamiczne części URL-a, np. `/users/{user_id}`, `/posts/{post_id}`

**Przykład:**

In [ ]:
@app.get("/users/{user_id}")
def get_user(user_id: int):
    """
    Pobierz użytkownika po ID

    - **user_id**: ID użytkownika (integer)
    """
    return {"user_id": user_id, "name": "John Doe"}

**Requesty:**
```bash
GET /users/42       → {"user_id": 42, "name": "John Doe"}
GET /users/abc      → Error 422 (validation error - oczekiwano int)
```

**Type hints = automatyczna walidacja:**
- `user_id: int` - FastAPI sprawdzi czy to liczba
- Jeśli nie - zwróci błąd 422 Unprocessable Entity
- Zero dodatkowego kodu walidacji!

#### Enum path parameters (ograniczone wartości):

In [ ]:
from enum import Enum

class ModelName(str, Enum):
    gpt3 = "gpt-3"
    gpt4 = "gpt-4"
    claude = "claude"

@app.get("/models/{model_name}")
def get_model(model_name: ModelName):
    """Tylko dozwolone modele"""
    if model_name == ModelName.gpt4:
        return {"model": model_name, "description": "GPT-4"}
    return {"model": model_name}

**Requesty:**
```bash
GET /models/gpt-4       → OK
GET /models/llama       → Error 422 (nie ma w Enum)
```

**Kiedy używać:**
- Ograniczona lista wartości (kategorie, typy, statusy)
- Dokumentacja automatycznie pokazuje dostępne opcje

### 2.4. Query Parameters (parametry zapytania)

**Definicja:**
Parametry po `?` w URL, np. `/items?skip=0&limit=10`

**Przykład:**

In [ ]:
@app.get("/items")
def get_items(skip: int = 0, limit: int = 10):
    """
    Pobierz listę elementów z paginacją

    - **skip**: Ile elementów pominąć (default: 0)
    - **limit**: Maksymalna liczba elementów (default: 10)
    """
    return {"skip": skip, "limit": limit, "items": []}

**Requesty:**
```bash
GET /items                  → skip=0, limit=10 (defaults)
GET /items?skip=20          → skip=20, limit=10
GET /items?skip=20&limit=5  → skip=20, limit=5
GET /items?limit=abc        → Error 422 (validation error)
```

#### Opcjonalne vs wymagane parametry:

In [ ]:
from typing import Optional

@app.get("/search")
def search(
    q: str,                    # WYMAGANY (brak default)
    category: Optional[str] = None,  # OPCJONALNY (default None)
    limit: int = 10           # OPCJONALNY (default 10)
):
    """Wyszukiwanie z filtrami"""
    return {"q": q, "category": category, "limit": limit}

**Requesty:**
```bash
GET /search                     → Error 422 (brak wymaganego 'q')
GET /search?q=fastapi           → OK (q="fastapi", category=None, limit=10)
GET /search?q=python&category=web → OK (wszystkie parametry)
```

#### Boolean query parameters:

In [ ]:
@app.get("/items")
def get_items(active: bool = True):
    """Filtruj po statusie aktywności"""
    return {"active": active}

**Requesty:**
```bash
GET /items?active=true     → active=True
GET /items?active=false    → active=False
GET /items?active=1        → active=True
GET /items?active=0        → active=False
GET /items?active=yes      → Error 422
```

**FastAPI rozpoznaje:** `true`, `false`, `1`, `0`, `True`, `False`, `yes`, `no`, `on`, `off`

### 2.5. Mieszanie Path i Query Parameters

**Przykład realistyczny:**

In [ ]:
@app.get("/users/{user_id}/posts")
def get_user_posts(
    user_id: int,              # PATH parameter
    published: bool = True,    # QUERY parameter
    limit: int = 10,          # QUERY parameter
    skip: int = 0             # QUERY parameter
):
    """
    Pobierz posty użytkownika

    - **user_id**: ID użytkownika (path)
    - **published**: Czy pokazać tylko opublikowane (query)
    - **limit**: Maksymalna liczba postów (query)
    - **skip**: Ile postów pominąć (query)
    """
    return {
        "user_id": user_id,
        "published": published,
        "limit": limit,
        "skip": skip,
        "posts": []
    }

**Requesty:**
```bash
GET /users/42/posts
  → user_id=42, published=True, limit=10, skip=0

GET /users/42/posts?published=false&limit=5
  → user_id=42, published=False, limit=5, skip=0
```

**Zasada:**
- **Path parameters** są ZAWSZE wymagane (są częścią URL)
- **Query parameters** mogą być opcjonalne (mają defaults lub Optional)

### 2.6. Kolejność deklaracji routes (WAŻNE!)

**❌ ZŁE - nigdy nie zadziała!**

In [ ]:
# ❌ ZŁE
@app.get("/users/{user_id}")
def get_user(user_id: int):
    return {"user_id": user_id}

@app.get("/users/me")
def get_current_user():
    return {"user": "current_user"}

**Problem:**
- Request `GET /users/me` zostanie złapany przez `/users/{user_id}`
- FastAPI spróbuje przekonwertować `"me"` na `int` → błąd 422

**✅ DOBRE - specificzne przed ogólnymi:**

In [ ]:
# ✅ DOBRE
@app.get("/users/me")
def get_current_user():
    return {"user": "current_user"}

@app.get("/users/{user_id}")
def get_user(user_id: int):
    return {"user_id": user_id}

**Zasada:**
Zawsze deklaruj bardziej **specyficzne** ścieżki przed **ogólnymi** (z path parameters)

### 2.7. Zwracanie odpowiedzi

#### Automatyczna serializacja do JSON:

In [ ]:
@app.get("/")
def root():
    return {"message": "Hello"}  # Automatycznie → JSON

**FastAPI automatycznie konwertuje:**
- `dict` → JSON object
- `list` → JSON array
- `str` → JSON string
- `int`, `float`, `bool` → odpowiednie typy JSON
- `None` → JSON null
- Pydantic models → JSON (później)

#### Różne formaty odpowiedzi:

In [ ]:
from fastapi.responses import HTMLResponse, PlainTextResponse

@app.get("/html", response_class=HTMLResponse)
def get_html():
    return "<h1>Hello HTML</h1>"

@app.get("/text", response_class=PlainTextResponse)
def get_text():
    return "Plain text response"

### 2.8. Dokumentacja w Swagger (docstrings)

In [ ]:
@app.get("/items/{item_id}")
def get_item(item_id: int, q: Optional[str] = None):
    """
    Pobierz element po ID

    Szczegółowy opis endpointu:
    - Zwraca element o podanym ID
    - Opcjonalne filtrowanie przez parametr q

    **Parametry:**
    - **item_id**: ID elementu (integer)
    - **q**: Opcjonalne query string dla filtrowania

    **Zwraca:**
    - JSON z danymi elementu
    """
    return {"item_id": item_id, "q": q}

**Efekt w Swagger UI:**
- Docstring funkcji → Description
- Markdown w docstring → Formatowanie
- Type hints → Automatyczne typy w dokumentacji

## 3. Best Practices

### ✅ Rób tak:

**1. Używaj type hints wszędzie:**

In [ ]:
# ✅ Dobre
@app.get("/items/{item_id}")
def get_item(item_id: int, limit: int = 10) -> dict:
    return {"item_id": item_id}

**2. Nazywaj endpointy RESTfully:**

In [ ]:
# ✅ Dobre - rzeczowniki + HTTP methods
@app.get("/users")           # Lista użytkowników
@app.post("/users")          # Nowy użytkownik
@app.get("/users/{id}")      # Konkretny użytkownik
@app.put("/users/{id}")      # Aktualizacja użytkownika
@app.delete("/users/{id}")   # Usunięcie użytkownika

**3. Dodawaj docstringi i descriptions:**

In [ ]:
# ✅ Dobre
@app.get("/items/{item_id}")
def get_item(item_id: int):
    """Pobierz element po ID"""
    return {"item_id": item_id}

**4. Grupuj powiązane endpointy z tags:**

In [ ]:
@app.get("/users", tags=["users"])
def get_users():
    pass

@app.post("/users", tags=["users"])
def create_user():
    pass

**5. Używaj Enum dla ograniczonych wartości:**

In [ ]:
# ✅ Dobre
class Status(str, Enum):
    active = "active"
    inactive = "inactive"

@app.get("/items/{status}")
def get_by_status(status: Status):
    pass

### ❌ Nie rób tak:

**1. Nie mieszaj konwencji nazewnictwa:**

```python
# ❌ Złe
@app.get("/get-users")       # Czasownik w URL
@app.get("/users/getAll")    # CamelCase + czasownik
```

**2. Nie używaj query params dla identyfikatorów:**

```python
# ❌ Złe
@app.get("/users?id=42")

# ✅ Dobre
@app.get("/users/42")
```

**3. Nie deklaruj specyficznych routes po ogólnych:**

```python
# ❌ Złe - nigdy nie zadziała
@app.get("/users/{user_id}")
def get_user(user_id: int):
    pass

@app.get("/users/me")  # Nigdy nie zostanie wywołane!
def get_current():
    pass
```

**4. Nie pomijaj type hints:**

```python
# ❌ Złe - brak walidacji
@app.get("/items/{item_id}")
def get_item(item_id, limit=10):
    pass

# ✅ Dobre
@app.get("/items/{item_id}")
def get_item(item_id: int, limit: int = 10):
    pass
```

**5. Nie używaj --reload w production:**

```bash
# ❌ Złe (development)
uvicorn main:app --reload

# ✅ Dobre (production)
uvicorn main:app --host 0.0.0.0 --port 8000 --workers 4
```